In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)
CHARTS_DIR = Path("charts")
CHARTS_DIR.mkdir(exist_ok=True)
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("../data/raw/rideops_support_tickets.csv")
for col in ["created_at", "first_response_at", "first_closed_at", "final_closed_at"]:
    df[col] = pd.to_datetime(df[col])

SNAPSHOT = pd.Timestamp("2026-09-30 23:59:00")
AVOIDABLE_OUTCOMES = [
    "Insufficient info - returned to L1",
    "Config issue - returned to L1",
    "Duplicate of known issue",
    "Not a bug - how-to",
]

df["first_close_hrs"] = (df["first_closed_at"] - df["created_at"]).dt.total_seconds() / 3600
df["final_close_hrs"] = (df["final_closed_at"] - df["created_at"]).dt.total_seconds() / 3600

print(f"Rows: {len(df)} | Date range: {df['created_at'].min().date()} to {df['created_at'].max().date()}")
print(df["status"].value_counts())

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
dates = pd.date_range("2026-07-05 23:59", "2026-09-30 23:59", freq="7D").tolist()
if dates[-1] < SNAPSHOT:
    dates.append(SNAPSHOT)

rows = []
for snap in dates:
    o = df[(df["created_at"] <= snap) & ((df["final_closed_at"].isna()) | (df["final_closed_at"] > snap))]
    rows.append({
        "date": snap.strftime("%b %d"),
        "Escalated": int((o["escalated_to_eng"] == "Yes").sum()),
        "L1 Only": int((o["escalated_to_eng"] == "No").sum()),
        "Total": int(len(o)),
    })
backlog_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(backlog_df["date"], backlog_df["Escalated"], color="#D9483B", label="Escalated to Eng")
ax.bar(backlog_df["date"], backlog_df["L1 Only"], bottom=backlog_df["Escalated"], color="#94A3B8", label="L1 Only")

for i, total in enumerate(backlog_df["Total"]):
    ax.text(i, total + 1.0, str(total), ha="center", va="bottom", fontweight="bold", fontsize=9)

ax.set_title("Open backlog jumped +60% (30 → 48); 71% of open tickets are stuck in escalation",
             fontsize=12.5, fontweight="bold", loc="left", pad=14)
ax.set_ylabel("Open Tickets (End of Week)")
plt.xticks(rotation=35, ha="right")
ax.legend(frameon=False, loc="upper left")
sns.despine()
plt.tight_layout()
plt.savefig(CHARTS_DIR / "01_backlog_and_volume_trend.png", dpi=300)
plt.show()

In [ ]:
esc = df[df["escalated_to_eng"] == "Yes"].copy()

# Chart 2: Avoidable vs Fixed Escalations
outcomes = esc["eng_outcome"].value_counts()
colors = ["#D9483B" if k in AVOIDABLE_OUTCOMES else ("#16A34A" if k == "Fixed" else "#94A3B8") for k in outcomes.index]

fig, ax = plt.subplots(figsize=(9.5, 4.6))
bars = ax.barh(outcomes.index[::-1], outcomes.values[::-1], color=colors[::-1])
for bar in bars:
    w = bar.get_width()
    ax.text(w + 4, bar.get_y() + bar.get_height()/2, f"{int(w)} ({w/len(esc)*100:.1f}%)",
            va="center", fontweight="bold", fontsize=9.5)

ax.set_xlim(0, outcomes.max() * 1.22)
ax.set_title("52.1% of engineering escalations (253/486) are avoidable before reaching eng",
             fontsize=12.5, fontweight="bold", loc="left", pad=14)
ax.set_xlabel("Escalated Tickets (Jul–Sep 2026)")
sns.despine()
plt.tight_layout()
plt.savefig(CHARTS_DIR / "02_escalation_outcomes_breakdown.png", dpi=300)
plt.show()

# Chart 3: Handoff Multiplier on Median Resolution Time
l1_med = df[df["escalated_to_eng"] == "No"]["final_close_hrs"].median()
h_meds = esc.groupby("eng_handoffs")["final_close_hrs"].median()

labels = ["L1 Only\n(0 handoffs)", "Escalated\n(1 handoff)", "Escalated\n(2 handoffs)", "Escalated\n(3 handoffs)"]
vals = [l1_med, h_meds.loc[1], h_meds.loc[2], h_meds.loc[3]]
bar_colors = ["#16A34A", "#F59E0B", "#EA580C", "#B91C1C"]

fig, ax = plt.subplots(figsize=(8.5, 4.5))
bars = ax.bar(labels, vals, color=bar_colors, width=0.55)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 4, f"{v:.1f} hrs\n({v/24:.1f} days)",
            ha="center", va="bottom", fontweight="bold", fontsize=9.5)

ax.set_ylim(0, max(vals) * 1.22)
ax.set_title("Every engineering bounce adds ~2.2 days to median resolution time",
             fontsize=12.5, fontweight="bold", loc="left", pad=14)
ax.set_ylabel("Median Time to Final Close (Hours)")
sns.despine()
plt.tight_layout()
plt.savefig(CHARTS_DIR / "03_handoffs_vs_resolution_time.png", dpi=300)
plt.show()

In [ ]:
cat_summary = df.groupby("category").agg(
    total=("ticket_id", "count"),
    open_sep30=("status", lambda s: (s != "Closed").sum()),
    escalated=("escalated_to_eng", lambda s: (s == "Yes").sum()),
    esc_rate_pct=("escalated_to_eng", lambda s: round((s == "Yes").mean() * 100, 1)),
    reopen_pct=("reopen_count", lambda s: round((s > 0).mean() * 100, 1)),
).sort_values("open_sep30", ascending=False)

display(cat_summary)

print("\nTop recurring 'Duplicate of known issue' descriptions (feed these into your Known-Issue Matcher):")
display(esc[esc["eng_outcome"] == "Duplicate of known issue"]["description"].value_counts().head(10))

In [ ]:
agent_df = df.groupby("assigned_agent").agg(
    tickets=("ticket_id", "count"),
    med_first_close=("first_close_hrs", "median"),
    reopen_pct=("reopen_count", lambda s: (s > 0).mean() * 100),
    esc_pct=("escalated_to_eng", lambda s: (s == "Yes").mean() * 100),
    csat=("csat_score", "mean"),
).reset_index()

fig, ax = plt.subplots(figsize=(8.8, 5.0))
for _, r in agent_df.iterrows():
    color = "#D9483B" if r["assigned_agent"] == "Karan B" else ("#F59E0B" if r["assigned_agent"] == "Rahul T" else "#475569")
    size = 180 if r["assigned_agent"] in ["Karan B", "Rahul T"] else 100
    ax.scatter(r["med_first_close"], r["reopen_pct"], s=size, color=color, zorder=3)
    offset_x = 1.0 if r["assigned_agent"] != "Rahul T" else -6.5
    ax.text(r["med_first_close"] + offset_x, r["reopen_pct"] + 0.6,
            f"{r['assigned_agent']} (CSAT {r['csat']:.2f})", fontsize=9, fontweight="bold" if size > 100 else "normal")

ax.set_title("Fast closures aren't quality: Karan B closes in 7.9h but drives 42% of all reopens",
             fontsize=12, fontweight="bold", loc="left", pad=14)
ax.set_xlabel("Median Time to First Close (Hours)")
ax.set_ylabel("Ticket Reopen Rate (%)")
ax.set_ylim(0, 35)
sns.despine()
plt.tight_layout()
plt.savefig(CHARTS_DIR / "04_agent_speed_vs_quality.png", dpi=300)
plt.show()

# Export verified baseline summary for Step 3 & Web App cross-check
eom_stats = []
for dt_str in ["2026-07-31 23:59:00", "2026-08-31 23:59:00", "2026-09-30 23:59:00"]:
    snap = pd.Timestamp(dt_str)
    o = df[(df["created_at"] <= snap) & ((df["final_closed_at"].isna()) | (df["final_closed_at"] > snap))]
    eom_stats.append({
        "date": dt_str[:10],
        "open_total": int(len(o)),
        "open_escalated": int((o["escalated_to_eng"] == "Yes").sum()),
        "open_gt_48h": int(((snap - o["created_at"]).dt.total_seconds() / 3600 > 48).sum()),
    })

summary = {
    "total_tickets": int(len(df)),
    "eom_backlog": eom_stats,
    "escalation_rate_pct": round(float((df["escalated_to_eng"] == "Yes").mean() * 100), 1),
    "avoidable_escalation_rate_pct": round(float(esc["eng_outcome"].isin(AVOIDABLE_OUTCOMES).mean() * 100), 1),
    "pingpong_rate_pct": round(float((esc["eng_handoffs"] >= 2).mean() * 100), 1),
    "reopen_rate_pct": round(float((df["reopen_count"] > 0).mean() * 100), 1),
    "avg_csat": round(float(df["csat_score"].mean()), 2),
    "csat_response_rate_pct": round(float(df["csat_score"].notna().mean() * 100), 1),
    "median_res_l1_hrs": round(float(df[df["escalated_to_eng"] == "No"]["final_close_hrs"].median()), 1),
    "median_res_esc_hrs": round(float(esc["final_close_hrs"].median()), 1),
}

with open(PROCESSED_DIR / "metrics_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved ../data/processed/metrics_summary.json")